In [0]:
%run ./lib/00_common

In [0]:
import numpy as np
import pandas as pd

run_date = get_text_widget("run_date", "2025-04-01")

orders_dir = str(Path(landing_orders_file(run_date)).parent)
customers_dir = str(Path(landing_customers_file(run_date)).parent)

clear_dir(orders_dir)
clear_dir(customers_dir)

seed = int(run_date.replace("-", ""))
rng = np.random.default_rng(seed)

n_orders = 1000
base_order_id = int(run_date.replace("-", "")) * 100000

orders_pdf = pd.DataFrame({
    "order_id": np.arange(base_order_id + 1, base_order_id + n_orders + 1, dtype=np.int64),
    "customer_id": rng.integers(1, 121, size=n_orders),
    "order_date": [run_date] * n_orders,
    "amount": np.round(rng.uniform(5, 500, size=n_orders), 2),
    "status": np.where(rng.random(n_orders) < 0.88, "PAID", "PENDING")
})

mask_negative = rng.random(n_orders) < 0.03
mask_null_customer = rng.random(n_orders) < 0.02

orders_pdf.loc[mask_negative, "amount"] = -10.0
orders_pdf.loc[mask_null_customer, "customer_id"] = None

extra_bad_pdf = pd.DataFrame([
    {"order_id": None, "customer_id": 3, "order_date": run_date, "amount": 25.0, "status": "PAID"},
    {"order_id": None, "customer_id": None, "order_date": run_date, "amount": -20.0, "status": "PAID"}
])

orders_pdf = pd.concat([orders_pdf, extra_bad_pdf], ignore_index=True)

customer_ids = np.arange(1, 121, dtype=np.int64)

customers_base_pdf = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_name": [f"customer_{i}" for i in customer_ids],
    "country": np.where(customer_ids % 3 == 0, "MX", np.where(customer_ids % 3 == 1, "CO", "CL")),
    "email": [f"user{i}@mail.com" for i in customer_ids],
    "updated_at": [f"{run_date}T08:00:00"] * len(customer_ids)
})

customers_updates_pdf = customers_base_pdf.loc[customers_base_pdf["customer_id"] <= 10].copy()
customers_updates_pdf["country"] = np.where(customers_updates_pdf["customer_id"] % 2 == 0, "AR", customers_updates_pdf["country"])
customers_updates_pdf["email"] = [f"updated_{i}@mail.com" for i in customers_updates_pdf["customer_id"]]
customers_updates_pdf["updated_at"] = f"{run_date}T10:00:00"

customers_pdf = pd.concat([customers_base_pdf, customers_updates_pdf], ignore_index=True)

write_csv(orders_pdf, landing_orders_file(run_date))
write_jsonl(customers_pdf, landing_customers_file(run_date))

print(f"Datos raw generados en Volume para run_date={run_date}")
print(f"orders_file    = {landing_orders_file(run_date)}")
print(f"customers_file = {landing_customers_file(run_date)}")
print(f"orders rows    = {len(orders_pdf)}")
print(f"customers rows = {len(customers_pdf)}")